# Исследование интерпретации ISCN-формул с помощью LLM



## 1. Что делает этот ноутбук

Сравниваются LLaMA-baseline и biomedical-модели. Каждый кейс представлен парой EN/RU с общим `pair_id`.

In [1]:
# Установка библиотек.
!pip -q install transformers accelerate bitsandbytes sentencepiece pandas numpy openpyxl scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00


## 2. Импорт библиотек

In [2]:
import os, re, json, math, torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

## 3. Загрузка данных

In [3]:
DATA_PATH = "iscn_dataset_full.xlsx"
SHEET_NAME = "adapted_dataset"
# from google.colab import files
# uploaded = files.upload(); DATA_PATH = next(iter(uploaded.keys()))
test_df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)


### 4. Настройка моделей

In [4]:
MODEL_CONFIGS = {
    "Llama-3.1-8B-Instruct": {"model_id": "meta-llama/Llama-3.1-8B-Instruct", "is_chat_model": True, "model_group": "general"},
    "BioGPT-Large": {"model_id": "microsoft/BioGPT-Large", "is_chat_model": False, "model_group": "bio"},
    "BioMistral-7B": {"model_id": "BioMistral/BioMistral-7B", "is_chat_model": True, "model_group": "bio"},
    "Meditron-7B": {"model_id": "epfl-llm/meditron-7b", "is_chat_model": False, "model_group": "bio"}
}
MODEL_GROUPS = {name: cfg["model_group"] for name, cfg in MODEL_CONFIGS.items()}
MODEL_CONFIGS

{'Llama-3.1-8B-Instruct': {'model_id': 'meta-llama/Llama-3.1-8B-Instruct',
  'is_chat_model': True,
  'model_group': 'general'},
 'BioGPT-Large': {'model_id': 'microsoft/BioGPT-Large',
  'is_chat_model': False,
  'model_group': 'bio'},
 'BioMistral-7B': {'model_id': 'BioMistral/BioMistral-7B',
  'is_chat_model': True,
  'model_group': 'bio'},
 'Meditron-7B': {'model_id': 'epfl-llm/meditron-7b',
  'is_chat_model': False,
  'model_group': 'bio'}}

###5. Единый шаблон промпта

In [5]:
def build_prompt(iscn_formula, language):
    if str(language).upper() == "EN":
        return f"""You are an expert cytogenetic consultant.
Interpret the following ISCN formula.
Do not invent facts. Explain only what follows from the notation.
Return a short, medically precise interpretation in English.

ISCN formula:
{iscn_formula}""".strip()
    return f"""Вы — эксперт-консультант по цитогенетике.
Интерпретируйте следующую формулу ISCN.
Не добавляйте вымышленных фактов. Объясняйте только то, что следует из записи.
Ответьте кратко и медицински точно на русском языке.

Формула ISCN:
{iscn_formula}""".strip()

test_df["prompt"] = test_df.apply(lambda r: build_prompt(r["iscn_formula"], r["language"]), axis=1)
test_df[["case_id", "language", "iscn_formula", "prompt"]].head()

,case_id,language,iscn_formula,prompt
0,case_001_en,EN,"46,XY,t(1;9)(q32;q13)",You are an expert cytogenetic consultant.\nInt...
1,case_001_ru,RU,"46,XY,t(1;9)(q32;q13)",Вы — эксперт-консультант по цитогенетике.\nИнт...
2,case_002_en,EN,"45,XY,der(13;14)(q10;q10)",You are an expert cytogenetic consultant.\nInt...
3,case_002_ru,RU,"45,XY,der(13;14)(q10;q10)",Вы — эксперт-консультант по цитогенетике.\nИнт...
4,case_003_en,EN,"45,XX,der(21;21)(q10;10)",You are an expert cytogenetic consultant.\nInt...


### 6. Функция загрузки модели

In [6]:
def load_model_and_tokenizer(model_id, use_4bit=True):
    quant_config = None
    if use_4bit:
        quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=quant_config, torch_dtype=torch.float16, trust_remote_code=True)
    model.eval()
    return tokenizer, model

## 7. Функция генерации ответа

In [7]:
def generate_answer(model, tokenizer, prompt, max_new_tokens=220):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05, pad_token_id=tokenizer.pad_token_id)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    if decoded.startswith(prompt): decoded = decoded[len(prompt):].strip()
    return decoded.strip()

### 8. Полный цикл генерации по набору данных

In [8]:
def run_inference_for_model(model_name, model_id, df, use_4bit=True):
    tokenizer, model = load_model_and_tokenizer(model_id, use_4bit=use_4bit)
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Inference: {model_name}"):
        rows.append({"model_name": model_name, "model_group": MODEL_GROUPS.get(model_name, "unknown"), "case_id": row["case_id"], "pair_id": row["pair_id"], "language": row["language"], "iscn_formula": row["iscn_formula"], "reference_interpretation": row["reference_interpretation"], "model_output": generate_answer(model, tokenizer, row["prompt"])})
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return pd.DataFrame(rows)

## 9. Запуск эксперимента по всем моделям

In [ ]:
# HF AUTH + MODEL LOADING

import os
import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer

HF_TOKEN = input("Enter your HF token: ").strip()
login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

def load_model_safe(model_id):
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        token=os.environ["HF_TOKEN"],
        trust_remote_code=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=os.environ["HF_TOKEN"],
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )

    model.eval()
    return tokenizer, model

In [10]:
def run_inference_for_model(model_name, model_id, df):
    print(f"\n=== {model_name} ===")

    try:
        tokenizer, model = load_model_safe(model_id)
    except Exception as e:
        print(f" Failed to load {model_id}: {e}")
        print(f"Skipping {model_name}")
        return pd.DataFrame()

    results = []

    for _, row in df.iterrows():
        try:
            prompt = row["question_en"]

            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=2048
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=200,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )

            answer = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[-1]:],
                skip_special_tokens=True
            )

            results.append({
                "model": model_name,
                "model_id": model_id,
                "language": "EN",
                "question": row["question_en"],
                "answer": answer
            })

        except Exception as e:
            print(f"Ошибка на строке: {e}")

    del model
    torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [12]:
def run_inference_for_model(model_name, model_id, df):
    print(f"\n=== {model_name} ===")

    try:
        # This function handles HF_TOKEN.
        tokenizer, model = load_model_safe(model_id)
    except Exception as e:
        print(f" Failed to load {model_id}: {e}")
        print(f"Skipping {model_name}")
        return pd.DataFrame()

    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Inference: {model_name}"):
        try:
            # Use the 'prompt' column generated earlier
            current_prompt = row["prompt"]

            inputs = tokenizer(
                current_prompt,
                return_tensors="pt",
                truncation=True,
                max_length=2048
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=200,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )

            answer = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[-1]:],
                skip_special_tokens=True
            ).strip()
            rows.append({
                "model_name": model_name,
                "model_group": MODEL_GROUPS.get(model_name, "unknown"),
                "case_id": row["case_id"],
                "pair_id": row["pair_id"],
                "language": row["language"], # Use actual language
                "iscn_formula": row["iscn_formula"],
                "reference_interpretation": row["reference_interpretation"],
                "prompt": current_prompt, # Store the actual prompt
                "model_output": answer
            })

        except Exception as e:
            print(f"Ошибка на строке при обработке ряда: {e}")
            # Optionally, append a row with an error message or skip it
            rows.append({
                "model_name": model_name,
                "model_group": MODEL_GROUPS.get(model_name, "unknown"),
                "case_id": row["case_id"],
                "pair_id": row["pair_id"],
                "language": row["language"],
                "iscn_formula": row["iscn_formula"],
                "reference_interpretation": row["reference_interpretation"],
                "prompt": current_prompt,
                "model_output": f"ERROR: {e}" # Indicate an error for this row
            })

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache() # Ensure cache is cleared
    return pd.DataFrame(rows)

all_results = []

for model_name, cfg in MODEL_CONFIGS.items():
    df_model = run_inference_for_model(
        model_name,
        cfg["model_id"],
        test_df
    )

    if not df_model.empty:
        all_results.append(df_model)
        print(f"✓ {model_name} done: {len(df_model)} rows")
    else:
        print(f"✗ {model_name} skipped")

if len(all_results) == 0:
    raise RuntimeError("No models worked")

results_df = pd.concat(all_results, ignore_index=True)
results_df.head()


=== Llama-3.1-8B-Instruct ===


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Inference: Llama-3.1-8B-Instruct:   0%|          | 0/1002 [00:00<?, ?it/s]

✓ Llama-3.1-8B-Instruct done: 1002 rows

=== BioGPT-Large ===


config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Inference: Meditron-7B:   0%|          | 0/1002 [00:00<?, ?it/s]

✓ Meditron-7B done: 1002 rows


,model_name,model_group,case_id,pair_id,language,iscn_formula,reference_interpretation,prompt,model_output
0,Llama-3.1-8B-Instruct,general,case_001_en,pair_001,EN,"46,XY,t(1;9)(q32;q13)",Type of rearrangement: balanced reciprocal tra...,You are an expert cytogenetic consultant.\nInt...,"del(9)(q13q22)[20]/46,XY[10]\n\n## Step 1: Bre..."
1,Llama-3.1-8B-Instruct,general,case_001_ru,pair_001,RU,"46,XY,t(1;9)(q32;q13)",Тип перестройки: сбалансированная реципрокная ...,Вы — эксперт-консультант по цитогенетике.\nИнт...,"del(9)(q13q21)[20]/46,XY[10]\n\nОписание:\nКар..."
2,Llama-3.1-8B-Instruct,general,case_002_en,pair_002,EN,"45,XY,der(13;14)(q10;q10)",Type of rearrangement: balanced Robertsonian t...,You are an expert cytogenetic consultant.\nInt...,"del(13)(q22q32),-17,+mar\n\n## Step 1: Break d..."
3,Llama-3.1-8B-Instruct,general,case_002_ru,pair_002,RU,"45,XY,der(13;14)(q10;q10)",Тип перестройки: сбалансированная Робертсоновс...,Вы — эксперт-консультант по цитогенетике.\nИнт...,pat\n\nОписание:\nКариотип человека с одной до...
4,Llama-3.1-8B-Instruct,general,case_003_en,pair_003,EN,"45,XX,der(21;21)(q10;10)","Karyotype: 45,XX,der(21;21)(q10;q10)\nType of ...",You are an expert cytogenetic consultant.\nInt...,pat\n\n## Step 1: Break down the ISCN formula ...


In [24]:
# === TABLE: expert + LLM answers EN/RU ===
from google.colab import files

os.makedirs("results", exist_ok=True)

# 1. Экспертные ответы EN/RU
expert_df = test_df.pivot_table(
    index=["pair_id", "iscn_formula"],
    columns="language",
    values="reference_interpretation",
    aggfunc="first"
).reset_index()

expert_df.columns.name = None
expert_df = expert_df.rename(columns={
    "EN": "expert_answer_en",
    "RU": "expert_answer_ru"
})

# 2. Вопросы EN/RU
questions_df = test_df.pivot_table(
    index=["pair_id", "iscn_formula"],
    columns="language",
    values="prompt",
    aggfunc="first"
).reset_index()

questions_df.columns.name = None
questions_df = questions_df.rename(columns={
    "EN": "question_en",
    "RU": "question_ru"
})

# 3. Ответы моделей (model + language)
llm_df = results_df.pivot_table(
    index=["pair_id", "iscn_formula"],
    columns=["model_name", "language"],
    values="model_output",
    aggfunc="first"
).reset_index()

# 4. flatten колонок
new_cols = []
for col in llm_df.columns:
    if isinstance(col, tuple):
        if col[1] == "":
            new_cols.append(col[0])
        else:
            model = str(col[0]).replace(" ", "_").replace("-", "_")
            lang = str(col[1]).lower()
            new_cols.append(f"llm_answer_{model}_{lang}")
    else:
        new_cols.append(col)

llm_df.columns = new_cols

# 5. объединение
final_answers = (
    questions_df
    .merge(expert_df, on=["pair_id", "iscn_formula"], how="left")
    .merge(llm_df, on=["pair_id", "iscn_formula"], how="left")
)

# 6. сохранение
output_path = "results/expert_vs_llm_answers.xlsx"
final_answers.to_excel(output_path, index=False)

print("Saved:", output_path)

files.download(output_path)

Saved: results/expert_vs_llm_answers.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Сохранение сырых результатов

In [25]:
os.makedirs("results", exist_ok=True)
if not results_df.empty:
    results_df.to_csv("results/raw_model_outputs.csv", index=False, encoding="utf-8-sig")
    results_df.to_excel("results/raw_model_outputs.xlsx", index=False)
    print("Сырые результаты сохранены")
else:
    print("results_df пустой: сначала запустите генерацию или загрузите готовые outputs.")

Сырые результаты сохранены


## 11. Автоматический расчёт Slot-Accuracy

In [16]:
def extract_iscn_slots(text):
    text = str(text)
    chromosomes = re.findall(r'(?<!\d)(\d{1,2})(?!\d)', text)
    arms = re.findall(r'(?<![A-Za-z])([pq])(?=\d)', text)
    loci = re.findall(r'([pq]\d+(?:\.\d+)?)', text)
    rearrangements = re.findall(r'(del|dup|inv|ins|t|der|add|i|r|mar)|[,+](\+?mar|\+?\d+)', text.lower())
    flat = []
    for x in rearrangements:
        flat.extend([i for i in x if i] if isinstance(x, tuple) else [x])
    return {"chromosomes": set(chromosomes), "arms": set(arms), "loci": set(loci), "rearrangements": set(flat)}

def slot_accuracy(iscn_formula, prediction):
    gold, pred = extract_iscn_slots(iscn_formula), extract_iscn_slots(prediction)
    scores = [len(gold[k] & pred[k]) / len(gold[k]) for k in gold if gold[k]]
    return float(np.mean(scores)) if scores else np.nan

print(slot_accuracy("46,XX,t(9;22)(q34;q11.2)", "Female karyotype with translocation between chromosomes 9 and 22 involving q34 and q11.2."))

0.9583333333333334


## 12. Автоматический Fact-score

Автоматический `Fact-score` — вспомогательный. Он не заменяет эксперта. Используются Slot-Accuracy, TF-IDF similarity с эталоном и покрытие ключевых понятий.

In [26]:
KEY_CONCEPTS = ["balanced","unbalanced","translocation","robertsonian","reciprocal","deletion","duplication","inversion","insertion","trisomy","monosomy","mosaic","marker","fertility","miscarriage","prenatal","pgt","сбаланс","несбаланс","транслокац","робертсон","реципрок","делец","дупликац","инверс","инсерц","трисом","моносом","мозаиц","маркер","фертиль","невынаш","пренаталь"]

def concept_coverage(reference, prediction):
    ref, pred = str(reference).lower(), str(prediction).lower()
    concepts = [c for c in KEY_CONCEPTS if c in ref]
    return np.nan if not concepts else sum(c in pred for c in concepts) / len(concepts)

def tfidf_similarity(reference, prediction):
    texts = [str(reference), str(prediction)]
    if not texts[0].strip() or not texts[1].strip(): return np.nan
    vec = TfidfVectorizer(ngram_range=(1,2), min_df=1).fit_transform(texts)
    return float(cosine_similarity(vec[0], vec[1])[0,0])

def automatic_fact_score(row):
    vals = [(row.get("slot_accuracy", np.nan), 0.35), (tfidf_similarity(row.get("reference_interpretation", ""), row.get("model_output", "")), 0.45), (concept_coverage(row.get("reference_interpretation", ""), row.get("model_output", "")), 0.20)]
    vals = [(s,w) for s,w in vals if pd.notna(s)]
    return np.nan if not vals else round(100 * sum(s*w for s,w in vals) / sum(w for s,w in vals), 2)

if not results_df.empty:
    results_df["slot_accuracy"] = results_df.apply(lambda r: slot_accuracy(r["iscn_formula"], r["model_output"]), axis=1)
    results_df["fact_score_auto"] = results_df.apply(automatic_fact_score, axis=1)


## 13. Подготовка шаблона экспертной оценки

In [18]:
if not results_df.empty:
    expert_template_df = results_df.copy()
    expert_template_df["fact_score_manual"] = np.nan
    expert_template_df["exact_match_manual"] = np.nan
    expert_template_df["expert_comment"] = ""
    expert_template_df.to_excel("results/expert_scoring_template.xlsx", index=False)
    print("Шаблон экспертной оценки сохранён: results/expert_scoring_template.xlsx")

Шаблон экспертной оценки сохранён: results/expert_scoring_template.xlsx


## 14. Раздельная подготовка оценок

В этом ноутбуке экспертный и автоматический `Fact-score` не смешиваются.

- `fact_score_auto` считается автоматически и может использоваться сразу для предварительной проверки.
- `fact_score_manual` остаётся пустым в шаблоне. Его нужно заполнить вручную позже.
- Финальная проверка гипотезы выполняется по явно выбранному режиму `FACT_SCORE_MODE`.




In [19]:
# Явный режим расчёта Fact-score.
# Сейчас оставляем auto, потому что экспертный fact_score_manual будет внесён вручную позже.

FACT_SCORE_MODE = "auto"  # "auto" или "manual"

EXPERT_SCORED_PATH = "results/expert_scoring_template_filled.xlsx"

if os.path.exists(EXPERT_SCORED_PATH):
    scored_df = pd.read_excel(EXPERT_SCORED_PATH)
elif "expert_template_df" in globals():
    scored_df = expert_template_df.copy()
else:
    scored_df = results_df.copy()

def prepare_score_columns(df, mode="auto"):
    df = df.copy()
    if "fact_score_auto" not in df.columns:
        df["fact_score_auto"] = np.nan
    if "fact_score_manual" not in df.columns:
        df["fact_score_manual"] = np.nan
    if "exact_match_manual" not in df.columns:
        df["exact_match_manual"] = np.nan
    if "expert_comment" not in df.columns:
        df["expert_comment"] = ""

    df["fact_score_auto"] = pd.to_numeric(df["fact_score_auto"], errors="coerce")
    df["fact_score_manual"] = pd.to_numeric(df["fact_score_manual"], errors="coerce")
    df["exact_match_manual"] = pd.to_numeric(df["exact_match_manual"], errors="coerce")

    if mode == "auto":
        df["fact_score_used"] = df["fact_score_auto"]
        df["fact_score_source"] = "auto_only"
    elif mode == "manual":
        df["fact_score_used"] = df["fact_score_manual"]
        df["fact_score_source"] = "manual_only"
    else:
        raise ValueError("FACT_SCORE_MODE должен быть 'auto' или 'manual'")

    missing_used = int(df["fact_score_used"].isna().sum())
    print(f"FACT_SCORE_MODE = {mode}")
    print(f"Строк без fact_score_used: {missing_used} из {len(df)}")
    if mode == "manual" and missing_used > 0:
        print("Внимание: в manual-режиме строки без fact_score_manual будут исключены из агрегированных расчётов.")

    return df

scored_df = prepare_score_columns(scored_df, FACT_SCORE_MODE)
scored_df[["case_id", "pair_id", "language", "model_name", "fact_score_auto", "fact_score_manual", "fact_score_used", "fact_score_source"]].head()


FACT_SCORE_MODE = auto
Строк без fact_score_used: 0 из 3006


,case_id,pair_id,language,model_name,fact_score_auto,fact_score_manual,fact_score_used,fact_score_source
0,case_001_en,pair_001,EN,Llama-3.1-8B-Instruct,48.35,NaN,48.35,auto_only
1,case_001_ru,pair_001,RU,Llama-3.1-8B-Instruct,39.69,NaN,39.69,auto_only
2,case_002_en,pair_002,EN,Llama-3.1-8B-Instruct,51.54,NaN,51.54,auto_only
3,case_002_ru,pair_002,RU,Llama-3.1-8B-Instruct,13.63,NaN,13.63,auto_only
4,case_003_en,pair_003,EN,Llama-3.1-8B-Instruct,51.26,NaN,51.26,auto_only


## 15. Расчёт RU/EN-Gap

RU/EN-Gap считается только по `fact_score_used`, то есть по явно выбранному режиму `FACT_SCORE_MODE`. Автоматический и экспертный Fact-score не агрегируются между собой.


In [20]:
def compute_ru_en_gap(scored_df, score_col="fact_score_used"):
    clean = scored_df.dropna(subset=[score_col]).copy()
    en = clean[clean["language"] == "EN"][["model_name", "model_group", "pair_id", score_col]].rename(columns={score_col: "fact_score_en"})
    ru = clean[clean["language"] == "RU"][["model_name", "pair_id", score_col]].rename(columns={score_col: "fact_score_ru"})
    merged = en.merge(ru, on=["model_name", "pair_id"], how="inner")
    merged["ru_en_gap"] = merged["fact_score_en"] - merged["fact_score_ru"]
    return merged

gap_df = compute_ru_en_gap(scored_df)



## 16. Агрегация итоговых метрик по моделям

Агрегация выполняется только по `fact_score_used`. Это предотвращает смешение ручной и автоматической оценки. Отдельные средние `fact_score_auto` и `fact_score_manual` выводятся только как диагностические поля.


In [21]:
def build_summary_tables(scored_df):
    clean = scored_df.dropna(subset=["fact_score_used"]).copy()
    summary = clean.groupby(["model_name", "model_group", "language"], as_index=False).agg(
        mean_fact_score=("fact_score_used", "mean"),
        mean_fact_score_auto=("fact_score_auto", "mean"),
        mean_fact_score_manual=("fact_score_manual", "mean"),
        mean_exact_match_manual=("exact_match_manual", "mean"),
        mean_slot_accuracy=("slot_accuracy", "mean"),
        n_cases=("case_id", "count")
    )
    gap_df = compute_ru_en_gap(scored_df, score_col="fact_score_used")
    gap_summary = gap_df.groupby(["model_name", "model_group"], as_index=False).agg(
        mean_ru_en_gap=("ru_en_gap", "mean"),
        median_ru_en_gap=("ru_en_gap", "median"),
        n_pairs=("pair_id", "count")
    )
    pivot = summary.pivot(index=["model_name", "model_group"], columns="language", values="mean_fact_score").reset_index().rename(columns={"EN": "Fact-score_EN", "RU": "Fact-score_RU"})
    exact = summary.pivot(index=["model_name", "model_group"], columns="language", values="mean_exact_match_manual").reset_index().rename(columns={"EN": "Exact_Match_EN_manual", "RU": "Exact_Match_RU_manual"})
    final_summary = pivot.merge(exact, on=["model_name", "model_group"], how="left").merge(gap_summary, on=["model_name", "model_group"], how="left")
    final_summary["fact_score_mode"] = FACT_SCORE_MODE
    return summary, gap_df, final_summary

summary_long, gap_df, final_summary = build_summary_tables(scored_df)



## 17. Проверка

In [22]:
def evaluate_hypothesis(final_summary, baseline_name="Llama-3.1-8B-Instruct"):
    rows = []
    for _, row in final_summary.iterrows():
        name, group = row["model_name"], row["model_group"]
        fact_en, gap = row.get("Fact-score_EN", np.nan), row.get("mean_ru_en_gap", np.nan)
        if group == "bio": gap_ok, criterion = gap > 30, "bio_gap_gt_30"
        elif name == baseline_name: gap_ok, criterion = gap < 20, "baseline_gap_lt_20"
        else: gap_ok, criterion = np.nan, "not_defined"
        rows.append({"model_name":name,"model_group":group,"Fact-score_EN":fact_en,"RU/EN_Gap":gap,"criterion_fact_en_ge_50":bool(fact_en >= 50) if pd.notna(fact_en) else False,"gap_criterion":criterion,"criterion_gap_ok":bool(gap_ok) if pd.notna(gap_ok) else False,"hypothesis_supported_for_model":bool((fact_en >= 50) and gap_ok) if pd.notna(fact_en) and pd.notna(gap_ok) else False})
    report = pd.DataFrame(rows)
    bio = final_summary[final_summary["model_group"] == "bio"]
    if not bio.empty:
        report = pd.concat([report, pd.DataFrame([{"model_name":"BIO_GROUP_MEAN","model_group":"bio_group","Fact-score_EN":bio["Fact-score_EN"].mean(),"RU/EN_Gap":bio["mean_ru_en_gap"].mean(),"criterion_fact_en_ge_50":bool(bio["Fact-score_EN"].mean() >= 50),"gap_criterion":"bio_group_gap_gt_30","criterion_gap_ok":bool(bio["mean_ru_en_gap"].mean() > 30),"hypothesis_supported_for_model":bool((bio["Fact-score_EN"].mean() >= 50) and (bio["mean_ru_en_gap"].mean() > 30))}])], ignore_index=True)
    return report

hypothesis_report = evaluate_hypothesis(final_summary)


## 18. Финальный блок анализа и экспорт

In [23]:

from google.colab import files

os.makedirs("results", exist_ok=True)

paths = {
    "summary_long": f"results/summary_long_{FACT_SCORE_MODE}.xlsx",
    "gap": f"results/ru_en_gap_by_case_{FACT_SCORE_MODE}.xlsx",
    "final": f"results/final_summary_{FACT_SCORE_MODE}.xlsx",
    "hypothesis": f"results/hypothesis_check_{FACT_SCORE_MODE}.xlsx",
    "scored": f"results/scored_outputs_separate_auto_manual_{FACT_SCORE_MODE}.xlsx"
}

summary_long.to_excel(paths["summary_long"], index=False)
gap_df.to_excel(paths["gap"], index=False)
final_summary.to_excel(paths["final"], index=False)
hypothesis_report.to_excel(paths["hypothesis"], index=False)
scored_df.to_excel(paths["scored"], index=False)

print("Файлы сохранены. Начинаю скачивание...")

for name, path in paths.items():
    try:
        print(f"⬇ Downloading {name}")
        files.download(path)
    except Exception as e:
        print(f" Ошибка скачивания {name}: {e}")





Файлы сохранены. Начинаю скачивание...
⬇ Downloading summary_long


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇ Downloading gap


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇ Downloading scored


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>